# 06 — Monitoring, Alerting, Scheduling, and Reporting

## Definition
Monitoring tracks model/data/pipeline health after training and deployment readiness.

## Theory
Stable operations require three views:
1. Data quality/drift
2. Model quality trend
3. Pipeline runtime and failures

## Motivation
Production ML systems fail silently without drift and runtime guardrails.

In [ ]:
from modules.data_generator import run_data_augmentation
from modules.feature_engineering import build_feature_pipeline
from modules.monitoring import run_data_drift_monitoring, pipeline_runtime_report, save_monitoring_snapshot
from modules.settings import load_config

cfg = load_config()
aug = run_data_augmentation(cfg)
feat = build_feature_pipeline(aug, cfg)

baseline = feat.iloc[: int(len(feat) * 0.7)]
current = feat.iloc[int(len(feat) * 0.7):]
num_cols = [c for c in feat.select_dtypes(include='number').columns if c != 'target_next_day']

drift = run_data_drift_monitoring(baseline, current, num_cols, cfg)
runtime = pipeline_runtime_report({'validation': 60.0, 'features': 90.0, 'training': 240.0, 'reporting': 45.0})

snapshot_path = save_monitoring_snapshot(
    metrics={'mae': 8.2, 'rmse': 12.6, 'r2': 0.81, 'mape': 18.4},
    drift_report=drift,
    runtime_report=runtime,
    config=cfg,
)
print('Monitoring snapshot:', snapshot_path)
drift

In [ ]:
import pandas as pd
from modules.data_loader import load_json

snap = pd.Series(load_json(snapshot_path))
snap

## Scheduling and Backfill Examples
```bash
# daily validation
airflow dags trigger 01_data_validation

# weekly training
airflow dags trigger 03_model_training

# backfill example
airflow dags backfill 01_data_validation --start-date 2026-06-01 --end-date 2026-06-07
```

## Operational Workflow
- if drift alert: inspect shifted features, evaluate retraining trigger
- if runtime alert: identify bottleneck task and optimize/parallelize
- if quality alert: block downstream model registration

## Production Recommendations
- move from SQLite to Postgres and Local/Celery executor
- externalize artifacts to object storage
- add SLA notifications to email/Slack
- enforce CI checks for DAG parse + notebook execution